In [1]:
import argparse
import pickle
import os
from pathlib import Path
import pandas as pd
import torch
import tqdm
from torch.utils.data import DataLoader
from torch.optim import Adam
import gc
from models.dataset import BERTDataset
from models.bert import BERT
from models.tokenizer import AsmTokenizer
import numpy as np
import json
from normalize_instr import normalize_instruction
data_dir = "."

In [2]:
def load_assembly_data(data):
    data_pairs = []
    for item in data:
        if item[0].startswith('0x'):
            temp = []
            for datum in item:
                instr=str.split(datum, '\t')[1:]
                instr = ' '.join(instr)
                temp.append(instr)
            item = temp
        normalized = [normalize_instruction(inst) for inst in item]
        if len(normalized) % 2 != 0:
            normalized.append('nop')
        for i in range(0, len(normalized)-1, 2):
            # Skip if either instruction is empty after normalization
            
                # Ensure both instructions are not empty
            data_pairs.append((normalized[i].strip(), normalized[i+1].strip()))
    return data_pairs

In [3]:
import json
for dataset_name in ["baseline-train", "baseline-valid"]:
    binary = os.path.join(data_dir, "outputs", f"{dataset_name}.pkl")
    with open(binary, 'rb') as c:
        data = pickle.load(c)
    data = load_assembly_data(data)
    output_path = os.path.join(data_dir, "outputs", f"{dataset_name}.jsonl")
    output_metadata_path = os.path.join(data_dir, "outputs", f"{dataset_name}-metadata.jsonl")
    with open(output_metadata_path, "w", encoding="utf-8") as f:
        metadata = {"__metadata__":{"dataset_size": len(data)}}
        f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
        
    with open(output_path, "w", encoding="utf-8") as f:
        for item in data:
            item = list(item)
            record = {"instr1": item[0], "instr2": item[1]}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
# dataset_name = "baseline-test"
# dataset_name = "data_split"

In [ ]:
import json
output_path = os.path.join(data_dir, "outputs", f"{dataset_name}.jsonl")
output_metadata_path = os.path.join(data_dir, "outputs", f"{dataset_name}_metadata.jsonl")
with open(output_metadata_path, "w", encoding="utf-8") as f:
    metadata = {"__metadata__":{"dataset_size": len(data)}}
    f.write(json.dumps(metadata, ensure_ascii=False) + "\n")
    
with open(output_path, "w", encoding="utf-8") as f:
    for item in data:
        record = {"instr1": item[0], "instr2": item[1]}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [1]:
from datasets import load_dataset
from models.collatefn import CollateFn
import os
import torch
from torch.utils.data import DataLoader
from models.tokenizer import AsmTokenizer
import numpy as np
import json
data_dir = "."
dataset_name = "baseline-train"
# dataset_name = "baseline-valid"
# dataset_name = "baseline-test"
dataset_path = os.path.join(data_dir, "outputs", f"{dataset_name}.jsonl")
metadata_path = os.path.join(data_dir, "outputs", f"{dataset_name}-metadata.jsonl")
with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)
    dataset_size = metadata['__metadata__']['dataset_size']
    print(f"Dataset size: {dataset_size}")

dataset = load_dataset('json', data_files=dataset_path, split='train', streaming=True)
dataset._info.dataset_size = dataset_size
shuffled_dataset = dataset.shuffle(seed=42, buffer_size=5000)
tokenizer = AsmTokenizer(vocab_file=os.path.join(data_dir, "outputs", f"baseline-vocab.txt"))

dataloader = DataLoader(
    shuffled_dataset,
    batch_size=32,           # 批次大小
    collate_fn=CollateFn(tokenizer)     # 使用自定义批处理函数
)
batch = next(iter(dataloader))
bert_input = batch['bert_input']
print("BERT input shape:", bert_input.shape)
bert_label = batch['bert_label']
print("BERT label shape:", bert_label.shape)

Dataset size: 69703068
Vocab loaded from .\outputs\baseline-vocab.txt
CollateFn time: 0.002791s
BERT input shape: torch.Size([32, 35])
BERT label shape: torch.Size([32, 35])


In [3]:
def random_word_parallel(tokens):
    # convert tokens to numpy array for random operations
    tokens_arr = np.array(tokens)
    
    # Randomly mask some tokens
    mask_prob = np.random.rand(len(tokens_arr)) < 0.15

    # generate a random strategy for each token
    # < 8: replace with <MASK>
    # == 8: replace with random token
    # > 8: keep original token
    strategy = np.random.randint(0, 10, size=len(tokens_arr))

    output = np.copy(tokens_arr)
    labels = np.zeros_like(tokens_arr)

    # get the mask token id
    mask_token_id = tokenizer.vocab['<MASK>']
    # get the random token ids
    random_tokens = np.random.randint(0, len(tokenizer.vocab), size=len(tokens_arr))
    # apply the masking strategy
    if np.any(mask_prob):
        # create boolean masks for each strategy
        mask_strategy = mask_prob & (strategy < 8)    # 80% MASK
        rand_strategy = mask_prob & (strategy == 8)   # 10% 随机词
        
        # apply the strategies
        output[mask_strategy] = mask_token_id
        output[rand_strategy] = random_tokens[rand_strategy]
        
        # set the labels
        labels[mask_prob] = tokens_arr[mask_prob]
    assert(len(output) == len(labels))
    return output.tolist(), labels.tolist()

In [4]:
import random
def random_word_loop(tokens):
    output = []
    labels = []
    for token in tokens:
        if random.random() < 0.15:
            random_choice = random.random()
            if random_choice < 0.8:
                output.append(tokenizer.vocab['<MASK>'])  # 80% Replace with MASK
            elif random_choice < 0.9:
                output.append(random.choice(list(tokenizer.vocab.values())))  # 10% Random token
            else:
                output.append(token)  # 10% Keep original
            labels.append(token)
        else:
            output.append(token)
            labels.append(0)
    assert(len(output) == len(labels))
    return output, labels

In [5]:
import time

tokens = [5, 55, 10, 33, 8, 5]
start = time.time()
output_parallel, labels_parallel = random_word_parallel(tokens)
end = time.time()
print(f"random_word_parallel time: {end - start:.6f}s")

start = time.time()
output_loop, labels_loop = random_word_loop(tokens)
end = time.time()
print(f"random_word_loop time: {end - start:.6f}s")

random_word_parallel time: 0.000000s
random_word_loop time: 0.000000s
